In [1]:
from pyspark.sql import SparkSession,Row,DataFrame
from delta.tables import DeltaTable
import pyspark.sql.functions as F
import os.path as path
import traceback
import os

spark = SparkSession \
    .builder \
    .appName("ConexaoPostgre") \
    .master("local[4]") \
    .config("spark.sql.warehouse.dir", "/Users/eduardoalberto/LoadFile/output/") \
    .getOrCreate()

sc = spark.sparkContext
spark.sparkContext.setLogLevel("OFF") 
print('PySpark Version :'+spark.version)
print('PySpark Version :'+spark.sparkContext.version)

spark

26/01/22 12:37:30 WARN Utils: Your hostname, MacBook-Pro-de-Eduardo.local resolves to a loopback address: 127.0.0.1; using 192.168.3.96 instead (on interface en0)
26/01/22 12:37:30 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Ivy Default Cache set to: /Users/eduardoalberto/.ivy2/cache
The jars for the packages stored in: /Users/eduardoalberto/.ivy2/jars
org.mongodb.spark#mongo-spark-connector_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-e83ff83a-f766-44c3-99ea-f44b3a21c8d0;1.0
	confs: [default]
	found org.mongodb.spark#mongo-spark-connector_2.12;10.3.0 in central


:: loading settings :: url = jar:file:/Users/eduardoalberto/opt/spark-3.5.4/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


	found org.mongodb#mongodb-driver-sync;4.8.2 in central
	[4.8.2] org.mongodb#mongodb-driver-sync;[4.8.1,4.8.99)
	found org.mongodb#bson;4.8.2 in central
	found org.mongodb#mongodb-driver-core;4.8.2 in central
	found org.mongodb#bson-record-codec;4.8.2 in central
:: resolution report :: resolve 1799ms :: artifacts dl 3ms
	:: modules in use:
	org.mongodb#bson;4.8.2 from central in [default]
	org.mongodb#bson-record-codec;4.8.2 from central in [default]
	org.mongodb#mongodb-driver-core;4.8.2 from central in [default]
	org.mongodb#mongodb-driver-sync;4.8.2 from central in [default]
	org.mongodb.spark#mongo-spark-connector_2.12;10.3.0 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   5   |   1   |   0   |   0   

PySpark Version :3.5.4
PySpark Version :3.5.4


In [2]:
CONF = {'inptFile': '/Users/eduardoalberto/LoadFile/input/kc_house_data.csv',
        'path_delta': '/Users/eduardoalberto/LoadFile/output/emp',
        'JDBC':  "jdbc:postgresql://localhost:5432/dbpostgres",
        'TABLE': "emp",
        'DB_USER': 'postgres',
        'DB_PASSWORD': 'postgre123'
}

t1 = spark.read.jdbc(CONF["JDBC"], CONF["TABLE"],properties={"user": CONF["DB_USER"],"password": CONF["DB_PASSWORD"],"driver": "org.postgresql.Driver"})

t1.show(5)

+----------+--------+--------+--------+---------+-----------+--------+------+----------+----+---------+-----+----------+-------------+--------+------------+-------+-------+--------+-------------+----------+---------+------------+
|        id| dt_date|   price|bedrooms|bathrooms|sqft_living|sqft_lot|floors|waterfront|view|condition|grade|sqft_above|sqft_basement|yr_built|yr_renovated|zipcode|    lat|    long|sqft_living15|sqft_lot15|id_number|dt_ref_carga|
+----------+--------+--------+--------+---------+-----------+--------+------+----------+----+---------+-----+----------+-------------+--------+------------+-------+-------+--------+-------------+----------+---------+------------+
|7129300520|20141013|221900.0|       3|      1.0|       1180|    5650|   1.0|         0|   0|        3|    7|      1180|            0|    1955|           0|  98178|47.5112|-122.257|         1340|      5650|        0|  2025-12-16|
|6414100192|20141209|538000.0|       3|     2.25|       2570|    7242|   2.0|   

In [2]:
def ingest(spark):

    CONF = {'inptFile': '/Users/eduardoalberto/LoadFile/input/kc_house_data.csv',
            'path_delta': '/Users/eduardoalberto/LoadFile/output/emp',
            'JDBC':  "jdbc:postgresql://localhost:5432/dbpostgres",
            'TABLE': "emp",
            'DB_USER': 'postgres',
            'DB_PASSWORD': 'postgre123'
        }
    try:
        if path.isfile(CONF["inptFile"]):
            
            kc_house_dt = spark.read.csv(CONF["inptFile"], header=True,inferSchema=True)\
                                    .withColumn("id_number", F.monotonically_increasing_id())\
                                    .withColumn("dt_ref_carga", F.current_date())
            
            dfs = kc_house_dt.select(   F.coalesce(F.col('id'),F.lit(0)).alias('id'),
                                        F.coalesce(F.col('dt_date'),F.lit(0)).alias('dt_date'),
                                        F.coalesce(F.col('price'),F.lit(0)).alias('price'),
                                        F.coalesce(F.col('bedrooms'),F.lit(0)).alias('bedrooms'),
                                        F.coalesce(F.col('bathrooms'),F.lit(0)).alias('bathrooms'),
                                        F.coalesce(F.col('sqft_living'),F.lit(0)).alias('sqft_living'),
                                        F.coalesce(F.col('sqft_lot'),F.lit(0)).alias('sqft_lot'),
                                        F.coalesce(F.col('floors'),F.lit(0)).alias('floors'),
                                        F.coalesce(F.col('waterfront'),F.lit(0)).alias('waterfront'),
                                        F.coalesce(F.col('view'),F.lit(0)).alias('view'),
                                        F.coalesce(F.col('condition'),F.lit(0)).alias('condition'),
                                        F.coalesce(F.col('grade'),F.lit(0)).alias('grade'),
                                        F.coalesce(F.col('sqft_above'),F.lit(0)).alias('sqft_above'),
                                        F.coalesce(F.col('sqft_basement'),F.lit(0)).alias('sqft_basement'),
                                        F.coalesce(F.col('yr_built'),F.lit(0)).alias('yr_built'),
                                        F.coalesce(F.col('yr_renovated'),F.lit(0)).alias('yr_renovated'),
                                        F.coalesce(F.col('zipcode'),F.lit(0)).alias('zipcode'),
                                        F.coalesce(F.col('lat'),F.lit(0)).alias('lat'),
                                        F.coalesce(F.col('long'),F.lit(0)).alias('long'),
                                        F.coalesce(F.col('sqft_living15'),F.lit(0)).alias('sqft_living15'),
                                        F.coalesce(F.col('sqft_lot15'),F.lit(0)).alias('sqft_lot15'),
                                        F.coalesce(F.col('id_number'),F.lit(0)).alias('id_number'),
                                        F.coalesce(F.col('dt_ref_carga'),F.lit('1900-01-01')).alias('dt_ref_carga'))



            dfs.write.format("parquet")\
                    .option("overwriteSchema", "true")\
                    .option("path",CONF["path_delta"])\
                    .partitionBy("dt_ref_carga")\
                    .mode("overwrite")\
                    .saveAsTable("default.tb_emp")
            # postgreSQL
            dfs.write.mode("overwrite").jdbc(CONF["JDBC"], CONF["TABLE"],properties={"user": CONF["DB_USER"],"password": CONF["DB_PASSWORD"],"driver": "org.postgresql.Driver"})

        else:
            print("arquivo não existe!")

    except Exception as e:
        print(f"Ocorreu o seguinte erro: {e}")
        traceback.print_exc()
    return dfs

df = ingest(spark)

df.count()

21613